# Test ToM Steering Vectors

This notebook loads and tests steering vectors from the `steering_vectors/` directory on Gemma-3-4B.

## 1. Setup & Imports

In [ ]:
import sys
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, Gemma3ForCausalLM, AutoConfig

# Add repeng to path
sys.path.insert(0, str(Path.cwd() / 'repeng'))
from repeng import ControlVector, ControlModel

print("✓ Imports successful!")

## 2. Load Model

In [ ]:
model_name = "google/gemma-3-4b-it"

print(f"Loading {model_name}...")

# Load config
config = AutoConfig.from_pretrained(model_name)

# Use bfloat16 for numerical stability
print("Using bfloat16 for better numerical stability...")

# Load model
if hasattr(config, 'vision_config'):
    print("Detected vision-language model. Loading text-only version...")
    base_model = Gemma3ForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
else:
    print("Loading standard causal LM...")
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token_id = 0

print(f"✓ Model loaded on {base_model.device}")
print(f"✓ Using dtype: {base_model.dtype}")

## 3. Wrap with ControlModel

In [ ]:
print("Setting up ControlModel...")

# Handle layer detection for multimodal vs standard architecture
if hasattr(base_model, 'language_model') and hasattr(base_model.language_model, 'layers'):
    print("Detected multimodal Gemma3 architecture")
    base_model.repeng_layers = base_model.language_model.layers
    num_layers = len(base_model.language_model.layers)
    base_model.config.num_hidden_layers = num_layers
elif hasattr(base_model, 'model') and hasattr(base_model.model, 'layers'):
    print("Detected standard architecture")
    base_model.repeng_layers = base_model.model.layers
    num_layers = len(base_model.model.layers)
else:
    raise ValueError("Could not find model layers!")

print(f"Total layers: {num_layers}")

# Using layers -4 to -19 (last 16 layers)
layer_ids = list(range(-4, -20, -1))
print(f"Wrapping layers: {layer_ids}")

model = ControlModel(base_model, layer_ids)

actual_layer_ids = [i if i >= 0 else num_layers + i for i in layer_ids]
print(f"Actual layer indices: {actual_layer_ids}")
print("✓ ControlModel ready!")

## 4. List Available Vectors

In [ ]:
import os

vector_dir = Path("steering_vectors")
vectors = sorted(vector_dir.glob("*.gguf"))

print(f"Found {len(vectors)} steering vectors:\n")
for i, v in enumerate(vectors, 1):
    print(f"{i:2d}. {v.name}")

## 5. Load a Steering Vector

In [ ]:
# Choose which vector to test
vector_name = "tom_procedural_forward_belief.gguf"  # Change this to test different vectors

vector_path = vector_dir / vector_name
print(f"Loading: {vector_name}")

steering_vector = ControlVector.import_gguf(str(vector_path))
print(f"✓ Vector loaded with {len(steering_vector.directions)} layer directions")

## 6. Test Generation Helper

In [ ]:
def generate_text(prompt, model, tokenizer, max_new_tokens=128, temperature=0.7, do_sample=False):
    """
    Generate text from the model.
    """
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    
    output = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.1
    )
    
    return tokenizer.decode(output[0], skip_special_tokens=True)

def generate_with_chat(user_message, model, tokenizer, max_new_tokens=128, temperature=0.7):
    """
    Generate using chat template format.
    """
    messages = [{"role": "user", "content": user_message}]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    input_ids = tokenizer(input_text, return_tensors="pt").to(model.device)
    
    output = model.generate(
        input_ids.input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=temperature,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.1
    )
    
    return tokenizer.decode(output[0], skip_special_tokens=True)

print("✓ Generation helpers ready")

## 7. Test the Steering Vector

### Classic False Belief Test

In [ ]:
test_prompt = """Sarah puts her toy in the red box and leaves the room.
While she's gone, John moves the toy to the blue box.
When Sarah returns, where will she look for her toy?

Answer:"""

print("=" * 80)
print("TEST: False Belief Scenario")
print("=" * 80)
print(f"\nPrompt: {test_prompt}\n")

# Baseline (no steering)
print("\n[BASELINE - No Steering]")
print("-" * 80)
model.reset()
baseline = generate_text(test_prompt, model, tokenizer, max_new_tokens=80, do_sample=False)
print(baseline)

# Positive steering
print("\n" + "=" * 80)
print("[POSITIVE STEERING - Coefficient: 1.5]")
print("-" * 80)
model.set_control(steering_vector, coeff=1000)
positive = generate_text(test_prompt, model, tokenizer, max_new_tokens=300, do_sample=False)
print(positive)

# Stronger positive steering
print("\n" + "=" * 80)
print("[POSITIVE STEERING - Coefficient: 3.0]")
print("-" * 80)
model.set_control(steering_vector, coeff=1500)
strong_positive = generate_text(test_prompt, model, tokenizer, max_new_tokens=300, do_sample=False)
print(strong_positive)

# Negative steering (anti-ToM)
print("\n" + "=" * 80)
print("[NEGATIVE STEERING - Coefficient: -2.0]")
print("-" * 80)
model.set_control(steering_vector, coeff=2000)
negative = generate_text(test_prompt, model, tokenizer, max_new_tokens=300, do_sample=False)
print(negative)

# Reset
model.reset()
print("\n" + "=" * 80)

### Test with Chat Template

In [ ]:
chat_prompt = """Sarah puts her toy in the red box and leaves the room.
While she's gone, John moves the toy to the blue box.
When Sarah returns, where will she look for her toy? Explain your reasoning."""

print("=" * 80)
print("TEST: Chat Format False Belief Scenario")
print("=" * 80)

# Baseline
print("\n[BASELINE]")
print("-" * 80)
model.reset()
baseline = generate_with_chat(chat_prompt, model, tokenizer, max_new_tokens=100)
print(baseline)

# With steering
print("\n" + "=" * 80)
print("[WITH STEERING - Coefficient: 2.0]")
print("-" * 80)
model.set_control(steering_vector, coeff=700)
steered = generate_with_chat(chat_prompt, model, tokenizer, max_new_tokens=400)
print(steered)

model.reset()
print("\n" + "=" * 80)

## 8. Custom Test Prompts

Try your own scenarios below:

In [ ]:
# Customize this section with your own test cases
custom_prompt = """what is your main speciality"""

coeff = 3000  # Adjust steering strength

print(f"Custom Test (coefficient: {coeff})")
print("=" * 80)

model.set_control(steering_vector, coeff=coeff)
result = generate_text(custom_prompt, model, tokenizer, max_new_tokens=120, do_sample=False)
print(result)

model.reset()

## 9. Compare Multiple Vectors

Load and compare different steering vectors on the same prompt:

In [ ]:
# Select vectors to compare
vectors_to_compare = [
    "tom_general_chat.gguf",
    "tom_forward_belief.gguf",
    "tom_belief_type.gguf"
]

test_prompt = """Sarah puts her toy in the red box and leaves the room.
While she's gone, John moves the toy to the blue box.
When Sarah returns, where will she look for her toy?

Answer:"""

coeff = 2.0

print("=" * 80)
print(f"VECTOR COMPARISON (coefficient: {coeff})")
print("=" * 80)
print(f"\nPrompt: {test_prompt}\n")

# Baseline
print("\n[BASELINE]")
print("-" * 80)
model.reset()
baseline = generate_text(test_prompt, model, tokenizer, max_new_tokens=80, do_sample=False)
print(baseline)

# Test each vector
for vec_name in vectors_to_compare:
    vec_path = vector_dir / vec_name
    if not vec_path.exists():
        print(f"\n[SKIPPED: {vec_name}] - File not found")
        continue
    
    print(f"\n{'=' * 80}")
    print(f"[{vec_name}]")
    print("-" * 80)
    
    vec = ControlVector.import_gguf(str(vec_path))
    model.set_control(vec, coeff=coeff)
    result = generate_text(test_prompt, model, tokenizer, max_new_tokens=80, do_sample=False)
    print(result)
    model.reset()

print("\n" + "=" * 80)

## 10. Coefficient Sweep

Test different steering strengths:

In [ ]:
# Load vector for sweep
vector_name = "tom_general_chat.gguf"
sweep_vector = ControlVector.import_gguf(str(vector_dir / vector_name))

test_prompt = """Sarah puts her toy in the red box and leaves the room.
While she's gone, John moves the toy to the blue box.
When Sarah returns, where will she look for her toy?

Answer:"""

coefficients = [0, 0.5, 1.0, 2.0, 3.0, 5.0]

print("=" * 80)
print(f"COEFFICIENT SWEEP: {vector_name}")
print("=" * 80)
print(f"\nPrompt: {test_prompt}\n")

for coeff in coefficients:
    print(f"\n{'=' * 80}")
    print(f"[Coefficient: {coeff}]")
    print("-" * 80)
    
    if coeff == 0:
        model.reset()
    else:
        model.set_control(sweep_vector, coeff=coeff)
    
    result = generate_text(test_prompt, model, tokenizer, max_new_tokens=80, do_sample=False)
    print(result)
    model.reset()

print("\n" + "=" * 80)

## Notes

**Vector Strength (coeff parameter):**
- Start with values between 0.5 and 3.0
- Positive values: Enhance ToM capabilities
- Negative values: Reduce ToM capabilities
- Typical good range: 1.5 to 2.5

**Available Vectors:**
- `tom_general_chat.gguf`: General ToM capabilities
- `tom_forward_belief.gguf`: Forward reasoning about beliefs
- `tom_backward_belief.gguf`: Backward reasoning about beliefs
- `tom_forward_action.gguf`: Action prediction
- `tom_belief_type.gguf`: True vs false belief handling
- `tom_core_capabilities.gguf`: Core ToM skills
- `tom_order_init.gguf`: Implicit vs explicit belief
- `tom_direction.gguf`: Forward vs backward reasoning
- `tom_variable.gguf`: Belief vs action distinction

**Tips:**
- Always call `model.reset()` after generation to clear steering
- Use `do_sample=False` for deterministic results
- Use `do_sample=True` with `temperature` for varied outputs
- Compare baseline vs steered to see the effect